# Sumário da configuração do switch Extreme X440-G2

## 1. Acesso inicial ao switch

O acesso inicial foi feito pela **serial console**, usando o **PuTTY**.

### O que foi necessário

* Conectar na porta **CONSOLE** do switch, não na USB e nem na MGMT.
* Usar o PuTTY em modo serial.
* Parâmetros seriais usados:

  * **9600**
  * **8 data bits**
  * **sem paridade**
  * **1 stop bit**
  * **flow control = XON/XOFF**

### Erro encontrado

**Terminal do PuTTY vazio**, sem aparecer login nem boot.

### Como foi resolvido

O problema estava no caminho físico/serial e na forma de acesso:

* o acesso correto era pela **console serial RJ45**;
* a **USB do switch não serve como console**;
* o PuTTY precisava estar configurado com os parâmetros seriais corretos;
* ao final, a comunicação serial passou a funcionar e foi possível entrar no CLI do switch.

---

## 2. Habilitação do SSH

Depois que o acesso serial funcionou, o próximo passo foi habilitar o SSH.

### Comandos usados

```bash
configure ssh2 key
enable ssh2
save configuration primary
```

### Erros encontrados

#### Erro 1

Ao executar:

```bash
configure ssh2 key
```

o switch mostrou:

> `Continue? (Y/N)`
> e foi respondido **No**.

### Como foi resolvido

Foi necessário executar novamente:

```bash
configure ssh2 key
```

e responder **Y** para a chave ser realmente gerada.

---

#### Erro 2

Depois foi tentado:

```bash
show ssh2
```

e o switch respondeu:

> `Incomplete command`

### Como foi resolvido

O comando correto para verificar o estado do SSH foi:

```bash
show management
```

Foi por esse comando que apareceu a confirmação de que o SSH estava ativo:

* `SSH access : Enabled`
* `Key valid`
* `tcp port 22`

---

## 3. Configuração do IP de gerenciamento

Depois do SSH, foi necessário configurar o IP da interface de gerenciamento do switch.

### Comandos usados

```bash
configure vlan mgmt ipaddress 192.168.1.10 255.255.255.0
configure iproute add default 192.168.1.1 vr vr-mgmt
save configuration primary
```

### O que cada um faz

* `configure vlan mgmt ipaddress ...`
  define o IP da interface de gerenciamento.
* `configure iproute add default ... vr vr-mgmt`
  define o gateway padrão da interface de gerenciamento.
* `save configuration primary`
  salva tudo para não perder após reboot.

---

## 4. Verificação do IP e da porta MGMT

Depois de configurar o IP, foi preciso confirmar se ele realmente estava aplicado e se a porta física de gerenciamento estava ativa.

### Comandos usados

```bash
show ipconfig mgmt
show vlan mgmt
show port mgmt
```

### Erros encontrados

#### Erro 1

Foi usado:

```bash
show ipconfig
```

mas a saída veio confusa e incompleta.

### Como foi resolvido

Foi usado o comando mais direto:

```bash
show ipconfig mgmt
```

que mostrou claramente o IP `192.168.1.10/24`.

---

#### Erro 2

A saída mostrou:

> `Mgmt-port on Mgmt is down`

### Como foi resolvido

Foi necessário ajustar a conexão física da porta **MGMT** até que o comando:

```bash
show port mgmt
```

passasse a mostrar o link como **ativo**, em **1000 FULL**.

---

## 5. Teste de conectividade a partir do notebook

Depois do IP e do link da porta MGMT estarem corretos, foi feito o teste a partir do notebook.

### Testes feitos

```bash
ping 192.168.1.10
ssh admin@192.168.1.10
```

### Erro encontrado

No Windows apareceu:

> `General failure`
> e depois o ping não respondia.

### Como foi resolvido

O problema era do lado do notebook/rede local:

* o notebook precisava estar na **mesma sub-rede** do switch;
* a porta física MGMT precisava estar ativa;
* depois que isso foi ajustado, a comunicação chegou ao ponto de tentar negociar o SSH.

---

## 6. Erro de compatibilidade do SSH

Quando a rede passou a funcionar, apareceu este erro no notebook:

```text
Unable to negotiate with 192.168.1.10 port 22:
no matching host key type found. They offer: ssh-rsa
```

### Motivo

O switch estava rodando uma versão antiga do ExtremeXOS e oferecendo apenas **`ssh-rsa`**.

### Como foi resolvido

A solução foi usar o cliente SSH em modo legado:

```bash
ssh -o HostKeyAlgorithms=+ssh-rsa admin@192.168.1.10
```

Se necessário, a forma mais completa seria:

```bash
ssh -o HostKeyAlgorithms=+ssh-rsa -o PubkeyAcceptedAlgorithms=+ssh-rsa admin@192.168.1.10
```

---

## 7. Verificação da versão do switch

Para entender por que o comando de algoritmo não funcionava, foi verificada a versão do firmware.

### Comando usado

```bash
show version
```

### Resultado

O switch estava em:

* **ExtremeXOS 21.1.1.4**
* build de **2016**

### Erro encontrado

Foi tentado:

```bash
configure ssh2 key algorithm rsa-sha2-256
```

e o switch respondeu:

> `Invalid input detected`

### Como foi resolvido

Foi identificado que essa versão do EXOS é antiga demais e **não suporta esse comando**.
As opções eram:

* usar SSH legado no notebook;
* ou atualizar o firmware do switch.

---

## 8. Comandos de espelhamento de porta

Como parte da configuração final, também foi mostrado como espelhar portas.

### Exemplo usado

Espelhar as portas **33 e 34** para a **35**:

```bash
create mirror CAPTURA
configure mirror CAPTURA to port 35
configure mirror CAPTURA add port 33 ingress-and-egress
configure mirror CAPTURA add port 34 ingress-and-egress
enable mirror CAPTURA
show mirror
save configuration primary
```

---

# Resumo final dos principais comandos

## Acesso e verificação

```bash
show management
show ipconfig mgmt
show vlan mgmt
show port mgmt
show version
```

## SSH

```bash
configure ssh2 key
enable ssh2
save configuration primary
```

## IP de gerenciamento

```bash
configure vlan mgmt ipaddress 192.168.1.10 255.255.255.0
configure iproute add default 192.168.1.1 vr vr-mgmt
save configuration primary
```

## Espelhamento

```bash
create mirror CAPTURA
configure mirror CAPTURA to port 35
configure mirror CAPTURA add port 33 ingress-and-egress
configure mirror CAPTURA add port 34 ingress-and-egress
enable mirror CAPTURA
show mirror
save configuration primary
```

## Acesso via notebook

```bash
ping 192.168.1.10
ssh -o HostKeyAlgorithms=+ssh-rsa admin@192.168.1.10
```

---

# Principais erros e correções

## 1. PuTTY vazio

**Causa:** acesso serial/cabo/configuração física.
**Solução:** usar a console serial correta com parâmetros corretos.

## 2. `configure ssh2 key` cancelado

**Causa:** resposta `No`.
**Solução:** rodar de novo e responder `Y`.

## 3. `show ssh2` incompleto

**Causa:** comando incorreto para essa versão.
**Solução:** usar `show management`.

## 4. MGMT sem link

**Causa:** porta física down.
**Solução:** ajustar cabo/conexão até `show port mgmt` ficar ativo.

## 5. `show ipconfig` confuso

**Causa:** comando genérico.
**Solução:** usar `show ipconfig mgmt`.

## 6. SSH moderno não conectava

**Causa:** switch antigo oferecendo `ssh-rsa`.
**Solução:** usar `ssh -o HostKeyAlgorithms=+ssh-rsa ...`.

## 7. `configure ssh2 key algorithm rsa-sha2-256` inválido

**Causa:** firmware antigo demais.
**Solução:** manter SSH legado temporariamente ou atualizar firmware.

---

